In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "processed"

lodes_jobs_path = JOBS_DIR / "lodes_jobs_2022_with_geometry.gpkg"
village_clean_path = BOUNDARIES_DIR / "village_clean.gpkg"

lodes_jobs_path, village_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/lodes_jobs_2022_with_geometry.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/village_clean.gpkg'))

In [2]:
lodes_jobs = gpd.read_file(lodes_jobs_path)
village = gpd.read_file(village_clean_path)

print("LODES CRS:", lodes_jobs.crs)
print("Village CRS:", village.crs)

LODES CRS: EPSG:2868
Village CRS: EPSG:2868


## CRS consistency

For **jobs per acre**, we need a **projected CRS** (meters or feet) so area calculations are meaningful.

We will:
- Choose `PROJECT_CRS` (e.g., same as parcels).
- Reproject LODES, TOC, and TOD layers if necessary.


In [3]:
# they are the same, but just because I am an anxious person...

PROJECT_CRS = "EPSG:2868"

if village.crs != PROJECT_CRS:
    village = village.to_crs(PROJECT_CRS)

lodes_jobs.crs, village.crs

(<Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich,
 <Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - 

In [4]:
if "jobs_total" not in lodes_jobs.columns and "C000" in lodes_jobs.columns:
    lodes_jobs["jobs_total"] = lodes_jobs["C000"].fillna(0)

if "area_sq_ft" not in lodes_jobs.columns:
    lodes_jobs["area_sq_ft"] = lodes_jobs.geometry.area
    lodes_jobs["area_acres"] = lodes_jobs["area_sq_ft"] / 43560.0

In [5]:
village.head(15)

,OBJECTID,NAME,geometry
0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598..."
1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551...."
2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860..."
3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89..."
4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89..."
5,21,Camelback East,"POLYGON ((676030.419 928829.819, 676010.366 92..."
6,22,Maryvale,"POLYGON ((628339.375 913042.563, 628422.563 91..."
7,23,Encanto,"POLYGON ((654775.813 910355.439, 655131.188 91..."
8,24,Alhambra,"POLYGON ((654743.25 928667, 654745.375 928399...."
9,25,North Mountain,"POLYGON ((623225.438 944219.75, 623293.625 949..."


In [6]:
# give villages a stable ID index:

village = village.reset_index().rename(columns={"index": "Village_ID"})

In [7]:
lodes_jobs.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,C000,CE01,CE02,CE03,area_sq_ft,area_acres,jobs_total,jobs_per_acre,geometry
0,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,"POLYGON ((827801.708 1091626.91, 827865.552 10..."
1,040130101021,04,013,010102,2,455,66,114,275,1.276018e+08,2929.334492,455,0.155325,"POLYGON ((725542.865 1023666.739, 723744.814 1..."
2,040130101021,04,013,010102,3,455,66,114,275,3.515639e+09,80707.955066,455,0.005638,"POLYGON ((712735.735 1097380.426, 712738.472 1..."
3,040130101021,04,013,010103,1,455,66,114,275,1.653396e+08,3795.674584,455,0.119873,"POLYGON ((744671.443 1002586.263, 744681.625 1..."
4,040130101021,04,013,010103,2,455,66,114,275,2.399420e+08,5508.309491,455,0.082602,"POLYGON ((765755.318 1002785.693, 765592.367 1..."


In [8]:
bg_x_village = gpd.overlay(
    lodes_jobs,
    village[["Village_ID", "NAME", "geometry"]],
    how="intersection",
    keep_geom_type=False
)

bg_x_village.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,C000,CE01,CE02,CE03,area_sq_ft,area_acres,jobs_total,jobs_per_acre,Village_ID,NAME,geometry
0,040130304011,04,013,061011,1,1748,389,537,822,1.503424e+08,3451.386805,1748,0.506463,6,Maryvale,"POLYGON ((581068.419 917531.353, 581093.415 91..."
1,040130304011,04,013,061011,2,1748,389,537,822,5.252857e+06,120.589005,1748,14.495517,6,Maryvale,"POLYGON ((575871.835 915038.49, 575866.841 914..."
2,040130304011,04,013,061013,1,1748,389,537,822,3.544408e+07,813.684034,1748,2.148254,6,Maryvale,"POLYGON ((575861.267 912423.451, 575868.581 91..."
3,040130405131,04,013,082002,1,191,49,72,70,1.283336e+07,294.613511,191,0.648307,6,Maryvale,"POLYGON ((588363.115 912655.943, 588864.54 912..."
4,040130405132,04,013,082002,2,75,19,27,29,8.139383e+06,186.854519,75,0.401382,6,Maryvale,"POLYGON ((591542.559 909444.735, 591538.635 90..."


In [ ]:
# Area of each BG–Village slice in square feet / acres
bg_x_village["intersect_sqft"] = bg_x_village.geometry.area
bg_x_village["intersect_acres"] = bg_x_village["intersect_sqft"] / 43560.0

# Fraction of each BG’s area that lies inside this Village
bg_x_village["area_weight"] = (
    bg_x_village["intersect_sqft"] /
    bg_x_village["area_sq_ft"]
)

# sanity check:

bg_x_village["area_weight"].describe()

count    1.331000e+03
mean     7.863022e-01
std      3.908200e-01
min      2.508484e-08
25%      9.144985e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: area_weight, dtype: float64

In [10]:
# area weight jobs into Villages

bg_x_village["jobs_contrib"] = (
    bg_x_village["jobs_total"].fillna(0) *
    bg_x_village["area_weight"]
)

In [11]:
village_jobs = (
    bg_x_village
    .groupby("Village_ID", as_index=False)
    .agg({
        "jobs_contrib": "sum",
        "intersect_acres": "sum"
    })
)

village_jobs = village_jobs.rename(columns={
    "jobs_contrib": "jobs_total",
    "intersect_acres": "acres_contributing"
})

# so total_jobs = total jobs allocated to that Village and/or station area
# acres_contributing is the total area in acres of block groups overlapping with the Village

In [12]:
villages_with_jobs = village.merge(
    village_jobs,
    on="Village_ID",
    how="left"
)

# true Village area based on its own geometry
villages_with_jobs["village_area_sqft"] = villages_with_jobs.geometry.area
villages_with_jobs["village_acres"] = villages_with_jobs["village_area_sqft"] / 43560.0

# jobs per acre
villages_with_jobs["jobs_per_acre"] = (
    villages_with_jobs["jobs_total"] / villages_with_jobs["village_acres"]
)

villages_with_jobs.head(15)

,Village_ID,OBJECTID,NAME,geometry,jobs_total,acres_contributing,village_area_sqft,village_acres,jobs_per_acre
0,0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598...",31854.214182,22832.880975,9.946003e+08,22832.880975,1.395103
1,1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551....",38138.709412,19579.686598,8.528911e+08,19579.686598,1.947871
2,2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860...",60064.332489,25481.856132,1.109990e+09,25481.856132,2.357141
3,3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89...",20731.840319,26503.624912,1.154498e+09,26503.624912,0.782227
4,4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89...",33900.231265,13600.592409,5.924418e+08,13600.592409,2.492555
5,5,21,Camelback East,"POLYGON ((676030.419 928829.819, 676010.366 92...",54700.185947,23242.629344,1.012449e+09,23242.629344,2.353442
6,6,22,Maryvale,"POLYGON ((628339.375 913042.563, 628422.563 91...",20996.374853,20820.532995,9.069424e+08,20820.532995,1.008446
7,7,23,Encanto,"POLYGON ((654775.813 910355.439, 655131.188 91...",18039.573404,6706.535758,2.921367e+08,6706.535758,2.689850
8,8,24,Alhambra,"POLYGON ((654743.25 928667, 654745.375 928399....",24520.718073,12315.416799,5.364596e+08,12315.416799,1.991059
9,9,25,North Mountain,"POLYGON ((623225.438 944219.75, 623293.625 949...",18837.398628,22212.332100,9.675692e+08,22212.332100,0.848060


In [13]:
villages_with_jobs["jobs_per_acre"].describe()

count    15.000000
mean      1.707191
std       1.007827
min       0.021092
25%       0.815143
50%       1.947871
75%       2.424848
max       3.503172
Name: jobs_per_acre, dtype: float64

In [14]:
total_jobs_bg = lodes_jobs["jobs_total"].sum()
total_jobs_village = villages_with_jobs["jobs_total"].sum()

total_jobs_village / total_jobs_bg

0.3348996275961671

In [15]:
village_jobs_path = JOBS_DIR / "village_jobs_2022.gpkg"
villages_with_jobs.to_file(village_jobs_path, driver="GPKG")

In [16]:
village_jobs_path = JOBS_DIR / "village_jobs_2022.csv"
villages_with_jobs.to_csv(village_jobs_path, index=False)